# Flink Dev Machine Deployment on AWS

## Prerequisites

- Python packages: boto3, paramiko, requests
- AWS credentials configured locally
- SSH key pair referenced in `deploy-config.yaml`

In [ ]:
# Version Information
from __future__ import annotations

NOTEBOOK_VERSION = '1.0'
LAST_UPDATED = '2026-03-21'

print(f'Notebook version: {NOTEBOOK_VERSION} (updated {LAST_UPDATED})')

## Prerequisites Setup

Install Python packages, import shared helpers, and verify AWS credentials via STS.

In [ ]:
# Package Installation
import subprocess
import sys

sys.path.insert(0, '.')

for pkg in ['boto3', 'paramiko', 'requests']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        )

print('\u2713 All required packages are available')

In [ ]:
# Import Required Packages
from datetime import datetime

import boto3
import paramiko
from deploy_common import allocate_eip
from deploy_common import create_instance
from deploy_common import install_icicle
from deploy_common import load_cluster_state
from deploy_common import load_deploy_config
from deploy_common import read_github_token
from deploy_common import resolve_maven_version
from deploy_common import resolve_ubuntu_ami
from deploy_common import save_cluster_state
from deploy_common import setup_logger
from deploy_common import setup_ubuntu
from deploy_common import ssh_exec
from deploy_common import update_ssh_config
from deploy_common import validate_aws_config
from deploy_common import wait_for_ssh

b3 = boto3.__version__
pm = paramiko.__version__
py = sys.version.split()[0]
print(f'boto3={b3} paramiko={pm} python={py}')
print('\u2713 All packages imported')

In [ ]:
# AWS Credential Verification
try:
    sts = boto3.client('sts', region_name='us-east-1')
    identity = sts.get_caller_identity()
    print(f'\u2713 Account: {identity["Account"]}')
    print(f'\u2713 ARN:     {identity["Arn"]}')
except Exception as e:
    print(f'\u2717 AWS credential check failed: {e}')
    raise

## Logger Setup

Configure a file-based logger for the deployment session.

In [ ]:
# Configure File-Based Logger
import logging

LOG_LEVEL = logging.INFO  # Change to logging.DEBUG for verbose output

logger, log_file = setup_logger('dev-machine', LOG_LEVEL)
print(f'\u2713 Logging to {log_file}')

## Configuration

Configuration is loaded from `deploy-config.yaml` (common + ingest-dev sections).

In [ ]:
# Configuration Variables
import random
import string

config = load_deploy_config('ingest-dev')

# --- Instance ID: load from state file or generate new ---
STATE_FILE = 'dev-machine-state.json'
_state = load_cluster_state(STATE_FILE, logger=logger)
if _state:
    INSTANCE_TAG = _state['instance_tag']
    print(f'Resuming existing instance: {INSTANCE_TAG}')
else:
    INSTANCE_TAG = ''.join(random.choices(string.ascii_lowercase, k=5))
    _state = {
        'instance_tag': INSTANCE_TAG,
        'created_at': datetime.now().isoformat(),
    }
    save_cluster_state(_state, STATE_FILE, logger=logger)
    print(f'New instance tag: {INSTANCE_TAG}')

# --- Convenience aliases ---
AWS_REGION = config['aws_region']
AWS_AZ = config['aws_az']
KEY_NAME = config['ssh_key_name']
KEY_PATH = config['ssh_key_path']
SECURITY_GROUP = config['security_group']
SSH_USER = config['ssh_user']
INSTANCE_TYPE = config['instance_type']
INSTANCE_NAME = f'{config["instance_name"]}-{INSTANCE_TAG}'
ROOT_VOLUME_SIZE = config.get('root_volume_size', 50)

logger.info('Configuration loaded')
print(f'Instance name: {INSTANCE_NAME}')
print(f'Instance type: {INSTANCE_TYPE}')
print(f'Region/AZ:     {AWS_REGION}/{AWS_AZ}')

In [ ]:
# Configuration Validation
validate_aws_config(KEY_PATH, SECURITY_GROUP, AWS_REGION, AWS_AZ)
print('\u2713 Configuration validated')

## Dynamic AMI and Maven Resolution

Query AWS for the latest Ubuntu 24.04 AMI and resolve the latest Maven
version from Maven Central.

In [ ]:
# Dynamic AMI and Maven Lookup
ec2 = boto3.client('ec2', region_name=AWS_REGION)

UBUNTU_AMI_ID, UBUNTU_AMI_NAME = resolve_ubuntu_ami(ec2, AWS_REGION, logger)
print(f'Ubuntu AMI: {UBUNTU_AMI_ID} ({UBUNTU_AMI_NAME})')

MAVEN_VERSION, MAVEN_URL = resolve_maven_version(logger=logger)
print(f'Maven:      {MAVEN_VERSION}')

## Deployment Steps

### Step 1: Launch Instance

Launch a single EC2 instance, allocate an Elastic IP, and update the local SSH config.

In [ ]:
# Step 1: Launch instance, allocate EIP, update SSH config
start_time = datetime.now()

instance_id = create_instance(
    ec2,
    INSTANCE_NAME,
    UBUNTU_AMI_ID,
    INSTANCE_TYPE,
    KEY_NAME,
    SECURITY_GROUP,
    AWS_AZ,
    root_volume_size=ROOT_VOLUME_SIZE,
    logger=logger,
)

eip_alloc, public_ip, private_ip, private_dns = allocate_eip(
    ec2,
    instance_id,
    INSTANCE_NAME,
    logger=logger,
)

dev_node = {
    'name': INSTANCE_NAME,
    'instance_id': instance_id,
    'eip_allocation': eip_alloc,
    'eip': public_ip,
    'private_ip': private_ip,
    'private_dns': private_dns,
}

# Update SSH config
ssh_entry = {
    'name': INSTANCE_NAME,
    'ip': public_ip,
    'user': SSH_USER,
    'key_path': KEY_PATH,
}
update_ssh_config(
    [ssh_entry],
    'dev-machine deployment instances',
    logger=logger,
)

# Save state
_state['dev_node'] = dev_node
save_cluster_state(_state, STATE_FILE, logger=logger)

logger.info(f'Instance {INSTANCE_NAME} launched: {public_ip}')
print(f'\u2713 Instance launched: {INSTANCE_NAME} ({public_ip})')

### Step 2: System Setup

Wait for SSH, then install system packages (Python 3.11, Java 11, Maven, AWS CLI, etc.).

In [ ]:
# Step 2: System setup
ssh = wait_for_ssh(
    public_ip,
    KEY_PATH,
    SSH_USER,
    timeout=300,
    logger=logger,
)
if not ssh:
    raise RuntimeError(f'SSH to {public_ip} timed out')
logger.info('SSH connected')

ok = setup_ubuntu(
    ssh,
    public_ip,
    MAVEN_VERSION,
    MAVEN_URL,
    SSH_USER,
    AWS_REGION,
    logger=logger,
)
if not ok:
    raise RuntimeError('setup_ubuntu failed')

logger.info('System setup complete')
print('\u2713 System packages installed')

### Step 3: Icicle & Ingest Setup

Clone the icicle repo, create a Python venv, install dependencies, then run
ingest-specific build steps (`make deps`). Note: `make compile` requires
`search.py` (Globus credentials) and is listed as a manual post-deployment step.

In [ ]:
import re

import requests

# Step 3: Icicle clone + venv + pip install, then ingest build steps
github_token = read_github_token()

ok = install_icicle(
    ssh,
    public_ip,
    github_token,
    config['git_username'],
    config['git_repo'],
    config['git_branch'],
    logger=logger,
)
if not ok:
    raise RuntimeError('install_icicle failed')
logger.info('Icicle installed')

# --- Ingest-specific setup ---
VENV = 'source ~/icicle/.venv/bin/activate'

# make deps — install platform-specific binary dependencies
logger.info('Running make deps')
ec, _, _ = ssh_exec(
    ssh,
    public_ip,
    f'{VENV} && cd ~/icicle/src/ingest && make deps',
    timeout=300,
    logger=logger,
)
if ec != 0:
    raise RuntimeError('make deps failed')

# Create resources directory
ssh_exec(
    ssh,
    public_ip,
    'mkdir -p ~/icicle/src/ingest/resources'
    ' && touch ~/icicle/src/ingest/resources/placeholder',
    logger=logger,
)

# Add custom deps path to Python site-packages
_pth_cmd = (
    f'{VENV} && cd ~/icicle/src/ingest'
    ' && echo "$PWD/my_deps"'
    ' > $(python -c "import site;'
    ' print(site.getsitepackages()[0])")/_custom.pth'
)
ssh_exec(ssh, public_ip, _pth_cmd, logger=logger)

# Install Flink S3 filesystem plugin for local PyFlink execution
# Dynamically resolve the latest 1.20.x version from Maven Central
_base = 'https://repo1.maven.org/maven2/org/apache/flink/flink-s3-fs-hadoop'
_listing = requests.get(f'{_base}/', timeout=10).text
_versions = re.findall(r'(1\.20\.\d+)/', _listing)
FLINK_S3_VERSION = sorted(
    _versions,
    key=lambda v: int(v.rsplit('.', 1)[1]),
)[-1]
FLINK_S3_JAR = f'flink-s3-fs-hadoop-{FLINK_S3_VERSION}.jar'
FLINK_S3_URL = f'{_base}/{FLINK_S3_VERSION}/{FLINK_S3_JAR}'
logger.info(f'Installing Flink S3 plugin {FLINK_S3_VERSION}')

_pyflink_lib = (
    "$(python -c 'import pyflink, os;"
    " print(os.path.dirname(pyflink.__file__))')/lib"
)
ec, _, _ = ssh_exec(
    ssh,
    public_ip,
    f'{VENV} && cd {_pyflink_lib} && wget -q {FLINK_S3_URL}',
    timeout=60,
    logger=logger,
)
if ec != 0:
    raise RuntimeError('Flink S3 plugin install failed')

logger.info('Ingest setup complete')
print('\u2713 Icicle cloned, venv created, deps installed')
print('\u2713 Ingest deps built (make deps)')
print(f'\u2713 Flink S3 plugin installed ({FLINK_S3_VERSION})')

## Deployment Summary

Print timing, instance details, SSH access command, and manual post-deployment steps.

In [ ]:
# Deployment Summary
end_time = datetime.now()
duration = end_time - start_time

# Save final state
_state['status'] = 'deployed'
_state['completed_at'] = end_time.isoformat()
save_cluster_state(_state, STATE_FILE, logger=logger)

ssh.close()

logger.info('=' * 60)
logger.info('DEPLOYMENT COMPLETED SUCCESSFULLY')
logger.info('=' * 60)
logger.info(f'Total deployment time: {duration}')
logger.info(f'Instance: {INSTANCE_NAME}')
logger.info(f'  Type:       {INSTANCE_TYPE}')
logger.info(f'  Public IP:  {public_ip}')
logger.info(f'  Private IP: {private_ip}')
logger.info(f'  Instance ID: {instance_id}')

print()
print('=' * 60)
print('DEPLOYMENT COMPLETE')
print('=' * 60)
print(f'Duration:     {duration}')
print(f'Instance:     {INSTANCE_NAME}')
print(f'Public IP:    {public_ip}')
print(f'Instance ID:  {instance_id}')
print()
print('SSH access:')
print(f'  ssh {INSTANCE_NAME}')
print()
print('Manual post-deployment steps:')
print(
    '  1. Create ~/icicle/src/ingest/search.py'
    ' with Globus Search credentials:',
)
print('       SEARCH_CLIENT_ID = "..."')
print('       SEARCH_CLIENT_SECRET = "..."')
print('  2. Compile after adding search.py:')
print('       cd ~/icicle/src/ingest && make compile')
print()
print('To run locally:')
print('  export IS_LOCAL=true')
_ak = 'aws configure get aws_access_key_id'
_sk = 'aws configure get aws_secret_access_key'
print(f'  export AWS_ACCESS_KEY_ID=$({_ak})')
print(f'  export AWS_SECRET_ACCESS_KEY=$({_sk})')
print('  cd ~/icicle/src/ingest && python main.py')
print()
print('  Note: AWS env vars are needed because the')
print('  Hadoop S3A connector reads credentials from')
print('  env vars, not ~/.aws/credentials.')